In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\death\MyPythonProjects\PythonProject1\ecommerce behavior data from multi category store\2019-Nov.csv", sep = ",")


print(df.shape)
df.head(5)

(67501979, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [2]:
df.columns #檢查是否有奇怪空格，沒有亂碼，命名一致

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='object')

In [3]:
df.info() #解析資料型態

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 4.5+ GB


In [4]:
import pandas as pd

df["event_time"] = pd.to_datetime(df["event_time"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[ns, UTC]
 1   event_type     object             
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  object             
 5   brand          object             
 6   price          float64            
 7   user_id        int64              
 8   user_session   object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(4)
memory usage: 4.5+ GB


In [5]:
df.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    21898171
brand             9224078
price                   0
user_id                 0
user_session           10
dtype: int64

In [6]:
df["category_code"] = df["category_code"].fillna("unknown")
df["brand"] = df["brand"].fillna("unkown")
df = df.dropna(subset = ["user_session"])

In [7]:
df_sample = df.sample(n = 1000000, random_state = 42)

In [8]:
df_sample.shape

(1000000, 9)

In [9]:
df_sample["event_type"].value_counts()

event_type
view        941622
cart         45037
purchase     13341
Name: count, dtype: int64

In [10]:
view = 941622
cart = 45037
purchase = 13341

cart_rate = cart / view
purchase_rate = purchase / view
cart_to_purchase = purchase / cart

cart_rate, purchase_rate, cart_to_purchase

(0.04782917136600462, 0.014168105672977055, 0.2962231054466328)

view -> cart = 4.78% \
view -> purchase = 1.42% \
cart -> purchase = 29.6%


重點：\
view -> cart 偏低，可能是因為價格、品牌、UI問題 \
轉換過低

In [11]:
#使用pandas寫法

counts = df_sample["event_type"].value_counts()

view = counts["view"]
cart = counts["cart"]
purchase = counts["purchase"]

cart_rate = cart / view
purchase_rate = purchase / view
cart_to_purchase = purchase / cart

round(cart_rate, 4), round(purchase_rate, 4), round(cart_to_purchase, 4)

(np.float64(0.0478), np.float64(0.0142), np.float64(0.2962))

In [12]:
#特徵工程
event_map = {        #行為強度 1弱,2中,3強
    "view" : 1,
    "cart" : 2,
    "purchase" : 3,
}
df_sample["event_score"] = df_sample["event_type"].map(event_map)

In [13]:
df_sample[["event_type", "event_score"]].head(5)

,event_type,event_score
10085233,view,1
17751200,view,1
26188885,cart,2
67483075,view,1
49466128,view,1


消費者相似度，找出相似的購物行為

In [14]:
user_vector = df_sample.pivot_table(
    index = "user_id",
    columns = "event_type",
    values = "event_score",
    aggfunc = "count",
    fill_value = 0
)

user_vector = user_vector[["view", "cart", "purchase"]]

In [15]:
user_vector.head(10)

event_type,view,cart,purchase
user_id,,,
31198833,2,0,0
62336140,1,0,0
63518127,1,0,0
80970791,1,0,0
107837897,1,0,0
120701478,1,0,0
122966408,1,0,0
126473256,1,0,0
147594972,1,0,0


In [16]:
# normalize 正常化

user_vector_norm = user_vector.div(user_vector.sum(axis = 1), axis = 0)

user_vector_norm.head(10)

event_type,view,cart,purchase
user_id,,,
31198833,1.0,0.0,0.0
62336140,1.0,0.0,0.0
63518127,1.0,0.0,0.0
80970791,1.0,0.0,0.0
107837897,1.0,0.0,0.0
120701478,1.0,0.0,0.0
122966408,1.0,0.0,0.0
126473256,1.0,0.0,0.0
147594972,1.0,0.0,0.0


In [17]:
user_vector_filter = user_vector[
    (user_vector["cart"] > 0) | (user_vector["purchase"] > 0)
]

user_vector_filter.head(10)

event_type,view,cart,purchase
user_id,,,
251171183,0,1,0
296197073,0,1,0
304707635,0,0,1
369251225,1,1,0
377743901,1,1,0
384989212,0,0,1
387887674,0,2,0
404851685,0,1,0
420986446,0,0,1


In [18]:
user_vector_norm2 = user_vector_filter.div(     #div df or serics資料除法
    user_vector_filter.sum(axis = 1),
    axis = 0
)

user_vector_norm2.head(10)

event_type,view,cart,purchase
user_id,,,
251171183,0.000000,1.000000,0.0
296197073,0.000000,1.000000,0.0
304707635,0.000000,0.000000,1.0
369251225,0.500000,0.500000,0.0
377743901,0.500000,0.500000,0.0
384989212,0.000000,0.000000,1.0
387887674,0.000000,1.000000,0.0
404851685,0.000000,1.000000,0.0
420986446,0.000000,0.000000,1.0


In [19]:
#計算余弦相似度

from sklearn.metrics.pairwise import cosine_similarity

sample_users = user_vector_norm2.head(100)

similarity_matrix = cosine_similarity(sample_users)

similarity_matrix[:5, :5]

array([[1.        , 1.        , 0.        , 0.70710678, 0.70710678],
       [1.        , 1.        , 0.        , 0.70710678, 0.70710678],
       [0.        , 0.        , 1.        , 0.        , 0.        ],
       [0.70710678, 0.70710678, 0.        , 1.        , 1.        ],
       [0.70710678, 0.70710678, 0.        , 1.        , 1.        ]])

In [20]:
import numpy as np

#假設找第0個user
target_user = 0

similarities = similarity_matrix[target_user]

#排序由大到小
similar_user_idx = np.argsort(similarities)[::-1] #argsort小到大排序 回傳用索引值 [::-1] 變成大到小

similar_user_idx[:20]

array([98, 96, 81, 88, 65, 87, 85, 80, 82, 73, 72, 23, 18,  6,  1, 10, 12,
       13, 14, 15])

In [21]:
similar_users = sample_users.iloc[similar_user_idx[:20]].index

similar_users

Index([512369745, 512368961, 512366358, 512367587, 512364515, 512367258,
       512366905, 512366040, 512366561, 512365722, 512365536, 468108437,
       460739110, 387887674, 296197073, 435741952, 441629183, 445372433,
       447770875, 448538520],
      dtype='int64', name='user_id')

In [22]:
similar_user_data = df_sample[
    df_sample["user_id"].isin(similar_users)
]

similar_user_data

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_score
64026337,2019-11-29 03:45:53+00:00,cart,14701435,2053013557133443581,furniture.living_room.cabinet,unkown,205.90,448538520,9ae1f858-01c2-4405-b108-23fd6efaf744,2
62171732,2019-11-27 20:00:23+00:00,cart,4804718,2053013554658804075,electronics.audio.headphone,apple,332.00,435741952,8011fc53-9e25-459a-bf70-6d6286f99749,2
29192529,2019-11-15 10:24:32+00:00,cart,28716475,2053013565782098913,apparel.shoes,respect,89.84,512365536,b4d94ff9-975e-4a11-ae2d-bbe0cef25395,2
47192770,2019-11-18 17:15:22+00:00,cart,1004249,2053013555631882655,electronics.smartphone,apple,795.90,447770875,0b5f083e-e7a2-4331-a881-3a904a73e7fe,2
20092121,2019-11-12 10:20:45+00:00,cart,26012611,2053013562778977083,unknown,unkown,7.16,460739110,182b0876-24d6-470d-8643-c98385413796,2
66371866,2019-11-30 09:32:06+00:00,cart,1005115,2053013555631882655,electronics.smartphone,apple,916.37,512366040,356a3cba-b0f7-4dfa-abaf-23690fe9df84,2
25795707,2019-11-14 19:19:19+00:00,cart,5500048,2053013563139687243,unknown,philips,115.81,512366561,c57836b8-de8e-40b4-9e03-c3a4ecea9403,2
66551294,2019-11-30 11:23:06+00:00,cart,11100284,2053013555036291451,appliances.personal.scales,scarlett,6.67,296197073,d0f682c7-4bf0-4048-91f0-27be1d53e605,2
26380468,2019-11-15 00:09:40+00:00,cart,4804056,2053013554658804075,electronics.audio.headphone,apple,159.33,512369745,4a2d9085-2bb1-4f29-97e9-800c37c33996,2
33832099,2019-11-16 05:53:56+00:00,cart,12705207,2053013553559896355,unknown,unkown,44.26,512367258,25deb85a-168c-4624-a7b0-9935e9feaad4,2


In [23]:
recommend_products = similar_user_data[
    similar_user_data["event_type"] == "purchase"
]

recommend_products

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_score


找出商品相似度，並商品推薦

In [24]:
top_product = df_sample["product_id"].value_counts().head(1000).index


df_small = df_sample[df_sample["product_id"].isin(top_product)]

df_small

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_score
10085233,2019-11-07 06:11:20+00:00,view,3701016,2053013565983425517,appliances.environment.vacuum,tefal,123.53,514543395,607cd696-afeb-4eb9-993c-7622fcb60cb8,1
17751200,2019-11-11 07:18:35+00:00,view,1004856,2053013555631882655,electronics.smartphone,samsung,127.38,528154368,ea191e6d-f34a-40d0-bca5-796ccaeaec02,1
26188885,2019-11-14 21:22:04+00:00,cart,1004227,2053013555631882655,electronics.smartphone,apple,997.11,529693105,4d205bdc-fff2-4515-9e0b-aa1c9958578e,2
24085226,2019-11-14 09:20:38+00:00,view,1004249,2053013555631882655,electronics.smartphone,apple,739.04,514138863,6e45539c-3094-4372-adbc-2d81c6076655,1
15419328,2019-11-10 03:09:06+00:00,view,1004840,2053013555631882655,electronics.smartphone,huawei,945.93,559265124,aab2575a-1a07-47eb-930a-33586fa007b4,1
...,...,...,...,...,...,...,...,...,...,...
35699086,2019-11-16 10:50:44+00:00,view,5100343,2053013553341792533,electronics.clocks,apple,334.35,537469708,887441f8-7b17-4d1a-a7ed-4d0a414f8238,1
26979205,2019-11-15 03:14:14+00:00,view,1004741,2053013555631882655,electronics.smartphone,xiaomi,190.22,571670262,59bc58d8-2297-4b43-8973-67a8cd2f9418,1
24377559,2019-11-14 11:47:10+00:00,cart,1004741,2053013555631882655,electronics.smartphone,xiaomi,190.22,514681594,f0f84247-f4a6-4945-a946-b7fd8191be80,2
54737049,2019-11-23 09:30:55+00:00,view,1003879,2053013555631882655,electronics.smartphone,huawei,566.04,567179202,1fa477e2-475b-4716-9bb5-937827d76094,1


In [25]:
item_matrix = df_small.pivot_table(
    index = "product_id",
    columns = "user_id",
    values = "event_score",
    aggfunc = "sum",
    fill_value = 0
)

item_matrix.shape

(1000, 323275)

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(item_matrix)

item_similarity.shape

(1000, 1000)

In [27]:
import numpy as np

item_ids = item_matrix.index.tolist()

target_item = 0

similarities = item_similarity[target_item]
similar_items_idx = np.argsort(similarities)[::-1]

product_info = df_sample[["product_id", "category_code", "brand"]]

top_items = [item_ids[i] for i in similar_items_idx[:10]]

result = product_info[
    product_info["product_id"].isin(top_items)
]

print(top_items)
result.sort_values("product_id")

[1002101, 1004836, 1004433, 1005186, 1005278, 1003768, 1004840, 1003712, 6301402, 8800093]


,product_id,category_code,brand
61543339,1002101,electronics.smartphone,samsung
20445226,1002101,electronics.smartphone,samsung
23291822,1002101,electronics.smartphone,samsung
23037414,1002101,electronics.smartphone,samsung
42219082,1002101,electronics.smartphone,samsung
...,...,...,...
32754120,8800093,electronics.telephone,nokia
24954311,8800093,electronics.telephone,nokia
56452371,8800093,electronics.telephone,nokia
49065078,8800093,electronics.telephone,nokia


In [28]:
def recommend_items(target_pid, top_n = 5):
    target_idx = item_ids.index(target_pid)
    similarities = item_similarity[target_idx]
    similar_item_idx = np.argsort(similarities)[::-1]
    similar_item_idx = similar_item_idx[1:top_n+1] #排除自己
    similar_items = [item_ids[i] for i in similar_item_idx]

    result = product_info[
        product_info["product_id"].isin(similar_items)
        ]

    return result


recommend_items(1002101)

,product_id,category_code,brand
6697678,1005186,electronics.smartphone,samsung
2768780,1004433,electronics.smartphone,samsung
52670702,1004836,electronics.smartphone,samsung
10874657,1004836,electronics.smartphone,samsung
63801387,1004836,electronics.smartphone,samsung
...,...,...,...
45205638,1005186,electronics.smartphone,samsung
6106565,1005186,electronics.smartphone,samsung
22427971,1004836,electronics.smartphone,samsung
45571669,1004836,electronics.smartphone,samsung


### <font color = "red">推薦系統評估(Evaluation)</font>

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\death\MyPythonProjects\PythonProject1\ecommerce behavior data from multi category store\2019-Nov.csv", sep = ",")


print(df.shape)
df.head(5)

(67501979, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [2]:
df.columns #檢查是否有奇怪空格，沒有亂碼，命名一致

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='object')

In [3]:
df.info() #解析資料型態

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 4.5+ GB


In [4]:
import pandas as pd

df["event_time"] = pd.to_datetime(df["event_time"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[ns, UTC]
 1   event_type     object             
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  object             
 5   brand          object             
 6   price          float64            
 7   user_id        int64              
 8   user_session   object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(4)
memory usage: 4.5+ GB


In [5]:
df.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    21898171
brand             9224078
price                   0
user_id                 0
user_session           10
dtype: int64

In [6]:
df["category_code"] = df["category_code"].fillna("unknown")
df["brand"] = df["brand"].fillna("unkown")
df = df.dropna(subset = ["user_session"])

In [7]:
df_sample = df.sample(n = 1000000, random_state = 42)

In [8]:
df_sample.shape

(1000000, 9)

In [9]:
df_sample["event_type"].value_counts()

event_type
view        941622
cart         45037
purchase     13341
Name: count, dtype: int64

### <font color = "red">建立商品對應user的表格</font>

In [29]:
item_matrix = df_small.pivot_table(
    index = "product_id",
    columns = "user_id",
    values = "event_score",
    aggfunc = "sum",
    fill_value = 0
)


print(item_matrix.shape)
item_matrix.head()

(1000, 323275)


user_id,31198833,80970791,107837897,120701478,183929790,197647707,210089363,214225869,226060852,226242984,...,579944251,579944855,579949794,579950058,579953281,579953464,579954487,579955312,579957484,579965752
product_id,,,,,,,,,,,,,,,,,,,,,
1002101,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002524,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002525,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002528,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002531,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### <font color = "red">相似度模型</font>

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(item_matrix)

print(item_similarity.shape)
item_similarity

(1000, 1000)


array([[1.00000000e+00, 7.29657260e-04, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [7.29657260e-04, 1.00000000e+00, 1.38937542e-02, ...,
        0.00000000e+00, 1.10241590e-03, 0.00000000e+00],
       [0.00000000e+00, 1.38937542e-02, 1.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.10241590e-03, 0.00000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00]],
      shape=(1000, 1000))

### <font color = "red">train test (80/20)切分法</font>

In [31]:
split_time = df_sample["event_time"].quantile(0.8)

train = df_sample[df_sample["event_time"] <= split_time]
test = df_sample[df_sample["event_time"] > split_time]

train.shape
test.shape

(200000, 10)

### <font color = "red">只保留「購買行為」purchase</font>

In [32]:
test_purchase = test[test["event_type"] == "purchase"]

test_purchase.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_score
62974484,2019-11-28 11:23:32+00:00,purchase,1002544,2053013555631882655,electronics.smartphone,apple,472.85,566701323,a626ef72-ed34-4f31-bb71-a355a8a719c0,3
57116194,2019-11-24 17:46:49+00:00,purchase,1004767,2053013555631882655,electronics.smartphone,samsung,246.96,572827859,be4bc425-fc85-4057-bf51-18a9c4ccd6ba,3
64242922,2019-11-29 06:05:53+00:00,purchase,1005135,2053013555631882655,electronics.smartphone,apple,1641.78,578089461,77a4622e-3866-40c5-8ff8-7a22341b9ff9,3
62333044,2019-11-28 03:38:36+00:00,purchase,1003306,2053013555631882655,electronics.smartphone,apple,612.19,558313698,c816160d-d70a-4542-b9e0-5e35b0a0eb89,3
61690690,2019-11-27 14:58:35+00:00,purchase,10800025,2053013554994348409,unknown,redmond,56.60,570662992,0f421448-670a-408e-9a80-8c48c30812d7,3


### <font color = "red">test答案</font>

In [33]:
test_user_items = test_purchase.groupby("user_id")["product_id"].apply(set)

test_user_items.head(10)

user_id
420986446             {1005235}
474414053             {2501685}
489085769           {100024154}
498248490             {1004904}
506710873             {1801883}
512372673    {5300977, 4803879}
512383452           {100013914}
512386086             {1005115}
512390221            {12600000}
512392999            {11800015}
Name: product_id, dtype: object

In [36]:
import numpy as np

item_ids = item_matrix.index.tolist() #把資料索引值，用串列方式表現(主要抓取product_id)

#recommend_for_user
def recommend_for_user(user_id, train_df, top_n=10):

    #使用test_user_items來挑選train_df裡面的user_id後再挑選product_id出來(不重複)
    user_items = train_df[
        train_df["user_id"] == user_id
    ]["product_id"].unique()

    scores = {}

    for pid in user_items:

        if pid not in item_ids: #檢查這product有沒有在item_ids裡
            continue

        idx = item_ids.index(pid)
        sim_scores = item_similarity[idx]

        for i, score in enumerate(sim_scores):
            sim_pid = item_ids[i]

            if sim_pid in user_items: #如果product已經被user買過，就跳過
                continue
            if score > 0:
                scores[sim_pid] = scores.get(sim_pid, 0) + score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [pid for pid, _ in ranked[:top_n]]

def precision_at_k(test_user_items, train_df, k=10): #準確率

    precisions = []

    for user in test_user_items.index:

        true_items = test_user_items[user]

        rec_items = recommend_for_user(user, train_df, top_n=k)

        if len(rec_items) == 0:
            continue

        hit = len(set(rec_items) & true_items)

        precisions.append(hit / k)

    return np.mean(precisions)

def recall_at_k(test_user_items, train_df, k = 10): #召回率

    recalls = []

    for user in test_user_items.index:

        true_items = test_user_items[user]

        rec_items = recommend_for_user(user, train_df, top_n=k)

        if len(rec_items) == 0:
            continue

        hit = len(set(rec_items) & true_items)

        recalls.append(hit / len(true_items))

    return np.mean(recalls)


#F1 = 2 * (precision * recall) / (precision + recall)
def f1_at_k(test_user_itmes, train_df, k = 10): #F1 score

    precisions = []
    recalls = []

    for user in test_user_items.index:

        true_items = test_user_items[user]

        rec_items = recommend_for_user(user, train_df, top_n=k)

        if len(rec_items) == 0:
            continue

        hit = len(set(rec_items) & true_items)

        precision = hit / k
        recall = hit / len(true_items)

        precisions.append(precision)
        recalls.append(recall)

    p = np.mean(precisions)
    r = np.mean(recalls)

    if p + r == 0:
        return 0

    return 2 * (p * r) / (p + r)

print("precision:", precision_at_k(test_user_items, train, k = 10))
print("recall:", recall_at_k(test_user_items, train, k = 10))
print("f1:", f1_at_k(test_user_items, train, k = 10))

precision: 0.02801358234295416
recall: 0.267402376910017
f1: 0.050714243896727365


In [37]:
print("precision:", precision_at_k(test_user_items, train, k = 3))
print("recall:", recall_at_k(test_user_items, train, k = 3))
print("f1:", f1_at_k(test_user_items, train, k = 3))

precision: 0.04923599320882852
recall: 0.1400679117147708
f1: 0.07286043838077763


In [38]:
print("precision:", precision_at_k(test_user_items, train, k = 20))
print("recall:", recall_at_k(test_user_items, train, k = 20))
print("f1:", f1_at_k(test_user_items, train, k = 20))

precision: 0.019100169779286927
recall: 0.3599320882852292
f1: 0.036275350443080216


k值增加

precision 分數越低 越不精準 推薦更多垃圾

recall 分數越高 越亂 更容易覆蓋答案

總結目前狀況\
抓得到部分答案\
但推薦不太精準